# 🧭 DocuAgent de campaña: entrenamiento, análisis y experimentación

El notebook exhaustivo: una **campaña completa** sobre el laboratorio, ejercitando cada capacidad del sistema y de su arquitectura de agentes, de principio a fin y en el orden en que se usan de verdad.

| Parte | Qué se ejercita | Coste |
|---|---|---|
| I · Preparación | estado de la memoria, eras, tracing | gratis |
| II · Entrenamiento | gimnasio con HITL, examen congelado, batch, reanudación | 💰💰 GPU+API (minutos) |
| III · Análisis | observatorio directo, DocuAgent analista, grounding | 💰 céntimos |
| IV · Experimentación | experimento dirigido, investigador con peaje, denegación, topes | 💰💰 al autorizar |
| V · Arquitectura a prueba | estabilidad, fallos, evaluación, fork de decisiones, costes | 💰 céntimos |
| VI · Cierre | checklist, higiene de la memoria | gratis |

**Requisitos**: arranque A1–A7 de `entrenar.ipynb` ejecutado; un PDF pequeño en `pdfs/`; secretos con las API keys (incluida `LANGSMITH_API_KEY` si quieres trazas). Las celdas con 💰💰 avisan antes de gastar.

---
# Parte I · Preparación

## I.1 · Tracing y estado inicial
Activamos LangSmith (opcional pero muy recomendable: cada etapa de la campaña quedará traceada y podrás hacer la autopsia de todo) y miramos qué sabe ya la memoria.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    os.environ["LANGSMITH_PROJECT"] = "campana-docuagent"
    print("Tracing -> proyecto 'campana-docuagent'")
except Exception as e:
    print(f"Sin tracing ({e}); la campaña funciona igual.")

# --- AJUSTA ---
PDF = Path("pdfs/test_13.pdf")            # documento principal de la campaña
CARPETA_PDFS = Path("pdfs")               # para la parte batch
assert PDF.exists(), f"no encuentro {PDF}"

from rag_lab import CONFIG, MetaSupervisor
from rag_lab.memory import Memoria
from rag_lab.corpus import hash_pdf
from rag_lab.analisis import observatorio as obs

mem = Memoria()
meta = MetaSupervisor()
print("\nPanorama inicial:", obs.panorama(mem))

## I.2 · Decisión de era
La trampa nº1 de la memoria: mezclar mediciones de escalas distintas (cada recalibración de la métrica cambia la escala). Decide AHORA tu política y sé consistente el resto de la campaña: o memoria limpia, o un `DESDE` que filtre todo el análisis.

In [ ]:
pan = obs.panorama(mem)
if pan.get("aviso"):
    print("⚠", pan["aviso"])
DESDE = None            # p. ej. "2026-07-06" = solo evidencia desde esa fecha
# Para empezar de cero (¡destructivo!): borra rag_memory/memoria_unificada.json
# desde Drive y re-ejecuta I.1. Los exámenes congelados también se pierden.
print("Política de era:", "todo" if DESDE is None else f"desde {DESDE}")

---
# Parte II · Entrenamiento 💰💰

## II.1 · Sesión completa con TODAS las pausas HITL
Una corrida real del gimnasio con las 4 pausas activas y un revisor que registra cada payload. Aquí ves al humano en el bucle del **entrenamiento**: revisar el examen, inyectar restricciones, decidir sobre la Fase B. Plan mínimo para que cueste poco.

In [ ]:
PAYLOADS = []

def revisor_campana(payload: dict) -> dict:
    PAYLOADS.append(payload)
    punto = payload.get("punto", "revisar_plan")
    print(f"\n{'='*60}\n[HITL] {punto}: {payload.get('mensaje', '')[:100]}")

    if punto == "tras_preguntas":
        print(f"  examen: {len(payload['preguntas'])} preguntas | "
              f"tasa verificación: {payload.get('tasa_verificacion')} | "
              f"congelado reutilizado: {payload.get('congelado_reutilizado')}")
        for q in payload["preguntas"][:3]:
            print(f"  [{q['i']}] {q['question'][:70]}")
        return {}                       # o {"vetar": [0]} para vetar la 0

    if punto == "tras_diagnostico":
        print("  inyecto restricción: solo 3 embeddings")
        return {"restricciones": {"embedding": {"etiquetas":
            ["bge-m3", "granite-311m-r2", "granite-97m-r2"]}}}

    if punto == "antes_fase_b":
        print(f"  pendiente de Fase B: {payload.get('pendiente_fase_b')}")
        return {}                       # {"saltar": True} la omitiría

    if punto == "tras_fase_b":
        print(f"  campeón: {payload.get('campeon')}")
        return {}                       # {"vetar": True} descartaría la coronación

    return {"accion": "aprobar"}


plan = {"bancos": ["embedding"], "n_runs": 2,
        "presupuesto": {"max_runs": 3, "max_preguntas": 40},
        "pausas": ["tras_preguntas", "tras_diagnostico", "antes_fase_b", "tras_fase_b"]}

stack = meta.ejecutar(str(PDF), plan=plan, revisor=revisor_campana, forzar=True)
print("\nSTACK:", {k: v for k, v in stack.items() if not k.startswith("_")})

## II.2 · El examen congelado, por dentro
El examen que acaba de usarse (o reutilizarse) está congelado por hash de contenido: cada corrida futura de este documento medirá contra las mismas preguntas. Inspecciónalo — es la base de toda comparabilidad.

In [ ]:
doc_hash = hash_pdf(str(PDF)) or ""
examen = mem.preguntas_congeladas(doc_hash) or []
print(f"Examen congelado: {len(examen)} preguntas (hash {doc_hash[:12]})\n")
for i, q in enumerate(examen[:5]):
    print(f"[{i}] {q.get('question', '')[:80]}")
    print(f"    cita: {q.get('source_context', '')[:80]}...\n")

## II.3 · Batch resumible sobre el corpus (opcional 💰💰)
El patrón de producción: bucle sobre la carpeta con `try` por documento — un fallo (Drive caído, API) pierde UN documento, no la tanda. La deduplicación por hash salta lo ya estudiado, así que re-ejecutar la celda REANUDA donde quedó.

In [ ]:
EJECUTAR_BATCH = False        # ponlo en True cuando quieras la tanda real

if EJECUTAR_BATCH:
    plan_batch = {"bancos": ["splitter", "embedding", "retriever"], "n_runs": 3,
                  "presupuesto": {"max_runs": 5, "max_preguntas": 40}}
    fallos = []
    for pdf in sorted(CARPETA_PDFS.glob("*.pdf")):
        print(f"\n########## {pdf.name} ##########")
        try:
            meta.ejecutar(str(pdf), plan=plan_batch)     # sin forzar: dedupe activo
        except Exception as e:
            fallos.append((pdf.name, str(e)[:120]))
            print(f"FALLO en {pdf.name}: {e}")
    print(f"\nTanda terminada. Fallos: {fallos or 'ninguno'}")
else:
    print("Batch desactivado (EJECUTAR_BATCH=False). La campaña sigue con lo que hay.")

## II.4 · Reanudación tras un fallo
Si una sesión casca DESPUÉS de la Fase A (los experimentos ya están en memoria), no se repite nada: `terminar_fase_b.py` retoma desde las certezas guardadas. Pruébalo en seco — sin `--persistir` solo informa.

In [ ]:
!python terminar_fase_b.py {PDF} --bancos embedding

---
# Parte III · Análisis 💰

## III.1 · El observatorio a pelo (sin agente, sin coste)
Antes del agente, su motor: la capa determinista. Todo lo que DocuAgent diga después debe poder rastrearse hasta una de estas llamadas.

In [ ]:
obs.informe(mem, desde=DESDE)

In [ ]:
print(obs.rendimiento(mem, "embedding", "bge-m3", desde=DESDE), "\n")
print(obs.duelo(mem, "embedding", "bge-m3", "granite-311m-r2", desde=DESDE), "\n")
print("victorias por perfil:", obs.victorias_por_perfil(mem, "embedding", desde=DESDE))
print("\ncorrelación densidad ~ bge-m3:",
      obs.correlacion_perfil(mem, "embedding", "bge-m3", "densidad", desde=DESDE))

## III.2 · DocuAgent analista
El agente canónico de `studio/graphs.py` — mismas tools, envueltas para el LLM. Pregúntale lo que acabas de calcular a mano y compara: la respuesta debe ser la prosa de esos números, no otra cosa.

In [ ]:
from langchain_core.messages import HumanMessage
from studio import graphs as G     # los grafos canónicos del proyecto
from langgraph.checkpoint.memory import MemorySaver

analista = G._c.compile(checkpointer=MemorySaver())

cfg = {"configurable": {"thread_id": "analisis-1"}}
r = analista.invoke({"messages": [HumanMessage(content=
    "¿Qué embedding recomienda la evidencia y con cuánta seguridad? "
    "Cita duelos y tamaños de muestra.")]}, cfg)
print(r["messages"][-1].content)

In [ ]:
# Multi-turno sobre el MISMO hilo: el agente recuerda su análisis anterior
r = analista.invoke({"messages": [HumanMessage(content=
    "¿Y ese veredicto cambia según el tipo de documento? Mira los patrones por perfil.")]},
    cfg)
print(r["messages"][-1].content)

## III.3 · Auditoría de grounding
El contrato dice que todo número sale de una tool. Verifícalo programáticamente sobre la conversación anterior — el anclaje del agente.

In [ ]:
import re

def auditar_grounding(resultado) -> None:
    fuentes = " ".join(str(m.content) for m in resultado["messages"]
                       if type(m).__name__ == "ToolMessage")
    final = resultado["messages"][-1].content
    nums = {n for n in re.findall(r"-?\d+(?:[.,]\d+)?", final)
            if len(n.replace('-', '').replace('.', '')) > 1
            or float(n.replace(',', '.')) > 9}
    sin_fuente = [n for n in nums if n not in fuentes
                  and n.replace(",", ".") not in fuentes]
    print("números:", sorted(nums) or "(ninguno)")
    print("⚠ SIN FUENTE:", sorted(sin_fuente)) if sin_fuente else print("✓ grounding OK")

auditar_grounding(r)

## III.4 · El anclaje humano cierra el triángulo
Máquina↔máquina está auditado (grounding); falta humano↔máquina: ¿la métrica con la que se midió todo coincide con TU juicio? Sesión corta del script de anclaje (interactiva: etiquetarás fragmentos).

In [ ]:
# Interactivo: descomenta cuando tengas 10 minutos para etiquetar.
# !python anclaje_humano.py {PDF} --n 5
# Y para re-analizar lo acumulado sin etiquetar más:
# !python anclaje_humano.py {PDF} --analizar-solo
print("Anclaje humano: ver anclaje_humano.py / anclaje_humano.ipynb")

---
# Parte IV · Experimentación 💰💰

## IV.1 · La capacidad, sin agente
La puerta acotada al laboratorio es una función normal: pruébala tú antes de dársela a un LLM. Topes: 1 banco, ≤4 etiquetas, ≤2 runs, examen congelado obligatorio.

In [ ]:
from rag_lab.experimento_dirigido import LIMITES, experimento_dirigido, plan_dirigido

print("Topes:", LIMITES, "\n")
print("Plan que construiría:", plan_dirigido("embedding", ["qwen3-0.6b", "bge-m3"]), "\n")

LANZAR = False        # True para gastar de verdad (1-3 min)
if LANZAR:
    res = experimento_dirigido(str(PDF), "embedding", ["qwen3-0.6b", "bge-m3"], n_runs=1)
    print(res)

## IV.2 · Los topes se defienden solos
Peticiones ilegales: banco inexistente, parrilla desmedida, etiqueta que no existe en el registro. Cada rechazo llega con mensaje accionable — que es EXACTAMENTE lo que recibirá el agente como observación cuando se equivoque.

In [ ]:
for descripcion, intento in [
    ("banco inexistente", lambda: plan_dirigido("magia", ["a"])),
    ("6 etiquetas > tope 4", lambda: plan_dirigido("embedding", list("abcdef"))),
    ("sin etiquetas", lambda: plan_dirigido("embedding", [])),
]:
    try:
        intento()
        print(f"✗ {descripcion}: NO se rechazó")
    except ValueError as e:
        print(f"✓ {descripcion} -> {e}")

## IV.3 · El investigador con peaje: el bucle del científico completo
DocuAgent analiza, formula hipótesis, propone experimento; el peaje te lo enseña con coste estimado; tú decides. Recompilamos con checkpointer (los grafos de Studio van sin él porque el servidor pone el suyo — sin checkpointer un interrupt no es reanudable).

In [ ]:
from langgraph.types import Command

investigador = G._ci.compile(checkpointer=MemorySaver())

cfg_inv = {"configurable": {"thread_id": "cientifico-1"}}
estado = investigador.invoke({"messages": [HumanMessage(content=
    f"Analiza la evidencia de qwen3-0.6b en embeddings. Si es insuficiente para "
    f"un veredicto, formula una hipótesis y propón un experimento acotado en "
    f"{PDF} para comprobarla.")]}, cfg_inv)

if estado.get("__interrupt__"):
    p = estado["__interrupt__"][0].value
    print("PEAJE:", p["mensaje"])
    for pet in p["peticiones"]:
        print("  ", pet)
else:
    print(estado["messages"][-1].content)

In [ ]:
# TU decisión (esta celda ES el formulario). Autorizar tarda minutos: es el lab.
if estado.get("__interrupt__"):
    r = input("¿Autorizas? (s/n) > ").strip().lower()
    estado = investigador.invoke(Command(resume={"seguir": True} if r == "s"
                                         else {"denegar": True}), cfg_inv)
print(estado["messages"][-1].content)

## IV.4 · Cerrar el bucle: re-análisis con la evidencia nueva
Si autorizaste, los resultados ya están en memoria. Pregunta al ANALISTA (el de solo lectura) y verás que su respuesta incorpora el experimento del investigador — dos agentes, una sola fuente de verdad.

In [ ]:
r = analista.invoke({"messages": [HumanMessage(content=
    "Con TODA la evidencia actual: ¿duelo entre qwen3-0.6b y bge-m3?")]},
    {"configurable": {"thread_id": "post-experimento"}})
print(r["messages"][-1].content)
auditar_grounding(r)

---
# Parte V · La arquitectura a prueba 💰

## V.1 · Fork de decisiones: autorizar Y denegar el mismo peaje
La demostración más potente del checkpointer: en el punto exacto del peaje, la historia se BIFURCA — una rama autoriza, otra deniega — y comparas ambos finales. Es responder '¿y si hubiera decidido lo contrario?' sin repetir nada.

In [ ]:
cfg_fork = {"configurable": {"thread_id": "fork-1"}}
estado = investigador.invoke({"messages": [HumanMessage(content=
    f"Propón un experimento en {PDF} con granite-97m-r2 y granite-311m-r2 "
    f"en embeddings para comparar tamaños.")]}, cfg_fork)

if estado.get("__interrupt__"):
    # checkpoint EXACTO de la pausa: el punto de bifurcación
    punto_pausa = investigador.get_state(cfg_fork).config

    # RAMA A: denegar (barata, la tomamos primera)
    rama_a = investigador.invoke(Command(resume={"denegar": True}), punto_pausa)
    print("=== RAMA DENEGADA ===\n", rama_a["messages"][-1].content[:400])

    # RAMA B: autorizar desde EL MISMO checkpoint (descomenta para gastar)
    # rama_b = investigador.invoke(Command(resume={"seguir": True}), punto_pausa)
    # print("\n=== RAMA AUTORIZADA ===\n", rama_b["messages"][-1].content[:400])
else:
    print("(el agente no propuso experimento; reformula la petición)")

## V.2 · Estabilidad: ¿instrumento o tómbola?
La misma pregunta 3 veces en hilos limpios. Ruta de tools y conclusión deben aguantar; si la conclusión baila, el agente no es fiable para esa pregunta.

In [ ]:
PREGUNTA = "¿Hay diferencia concluyente entre bge-m3 y granite-311m-r2 en embeddings?"
rutas, resps = [], []
for i in range(3):
    r = analista.invoke({"messages": [HumanMessage(content=PREGUNTA)]},
                        {"configurable": {"thread_id": f"estab-{i}"}})
    ruta = tuple(tc["name"] for m in r["messages"]
                 for tc in (getattr(m, "tool_calls", None) or []))
    rutas.append(ruta); resps.append(r["messages"][-1].content)
    print(f"pasada {i+1}: {' -> '.join(ruta) or '(sin tools)'}")
print("\nrutas idénticas:", len(set(rutas)) == 1)

## V.3 · Inyección de fallos: ¿degrada con honestidad?
Rompemos una tool y observamos: el buen agente cuenta el error y razona con lo que queda; el malo alucina el dato que no obtuvo.

In [ ]:
from langchain_core.tools import tool as _tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage

@_tool
def duelo_roto(banco: str, a: str, b: str) -> str:
    """Comparación PAREADA entre dos configs (única fuente para afirmar victorias)."""
    raise RuntimeError("memoria corrupta: colección de experimentos ilegible")

tools_f = [G.listar_documentos, G.rendimiento_de_etiqueta, duelo_roto,
           G.victorias_por_perfil]
from langchain_anthropic import ChatAnthropic
llm_f = ChatAnthropic(model="claude-haiku-4-5-20251001",
                      temperature=0).bind_tools(tools_f)

cf = StateGraph(MessagesState)
cf.add_node("agente", lambda e: {"messages": [llm_f.invoke(
    [SystemMessage(content=G.SISTEMA)] + e["messages"])]})
cf.add_node("tools", ToolNode(tools_f))
cf.add_edge(START, "agente")
cf.add_conditional_edges("agente", tools_condition)
cf.add_edge("tools", "agente")
fragil = cf.compile(checkpointer=MemorySaver())

r = fragil.invoke({"messages": [HumanMessage(content=PREGUNTA)]},
                  {"configurable": {"thread_id": "fallo-1"}})
print(r["messages"][-1].content)

## V.4 · Evaluación sistemática contra la verdad determinista
Tu filosofía de laboratorio aplicada al agente: el observatorio da la verdad de cada duelo, el agente aprueba si su prosa la respeta.

In [ ]:
def evaluar(banco, a, b):
    verdad = obs.duelo(mem, banco, a, b, desde=DESDE)
    r = analista.invoke({"messages": [HumanMessage(content=
        f"¿Diferencia concluyente entre {a} y {b} en {banco}? Cita el veredicto.")]},
        {"configurable": {"thread_id": f"eval-{a}-{b}"}})
    t = r["messages"][-1].content.lower()
    if not verdad.get("disponible"):
        ok, esp = ("insuficiente" in t or "sin" in t), "evidencia insuficiente"
    elif verdad["global"]["concluyente"]:
        ok, esp = ("concluyente" in t and " no concluyente" not in t), "concluyente"
    else:
        ok, esp = ("empate" in t or "no concluyente" in t or "no es concluyente" in t), "empate"
    return ok, esp, r["messages"][-1].content[:120]

bateria = [("embedding", "bge-m3", "granite-311m-r2"),
           ("embedding", "bge-m3", "voyage-3-lite"),
           ("embedding", "granite-97m-r2", "granite-311m-r2")]
aciertos = 0
for banco, a, b in bateria:
    ok, esp, resp = evaluar(banco, a, b)
    aciertos += ok
    print(f"{'✓' if ok else '✗'} {a} vs {b} (esperado: {esp})\n   {resp}...\n")
print(f"Batería: {aciertos}/{len(bateria)}")

## V.5 · Contabilidad: qué cuesta cada etapa
Un callback contador sobre una pregunta representativa de cada agente. Con estos números decides qué modelo darle a cada uno en el día a día.

In [ ]:
import time
from langchain_core.callbacks import BaseCallbackHandler

class Contador(BaseCallbackHandler):
    def __init__(self): self.llm = 0; self.tin = 0; self.tout = 0
    def on_chat_model_start(self, *a, **k): self.llm += 1
    def on_llm_end(self, resp, **k):
        try:
            u = resp.generations[0][0].message.usage_metadata or {}
            self.tin += u.get("input_tokens", 0); self.tout += u.get("output_tokens", 0)
        except Exception: pass

for nombre, grafo, preg in [("analista", analista, PREGUNTA),
                            ("investigador (sin autorizar)", investigador,
                             f"¿Merece la pena otro experimento sobre {PDF.name}?")]:
    c = Contador()
    est = grafo.invoke({"messages": [HumanMessage(content=preg)]},
                       {"configurable": {"thread_id": f"coste-{nombre[:4]}",
                                         "callbacks": [c]}})
    extra = " [pausado en peaje]" if est.get("__interrupt__") else ""
    print(f"{nombre:32} llamadas={c.llm} tokens in/out={c.tin}/{c.tout}{extra}")

---
# Parte VI · Cierre

## VI.1 · Checklist de la campaña

Marca lo ejercitado — si todo está en verde, has tocado cada capacidad del sistema:

- [ ] **Entrenamiento**: sesión con las 4 pausas HITL · examen congelado inspeccionado · batch resumible · reanudación de Fase B desde memoria
- [ ] **Análisis**: observatorio directo (informe, duelos, perfiles, correlaciones) · analista multi-turno · grounding verificado · anclaje humano
- [ ] **Experimentación**: capacidad directa sin agente · topes defendiéndose · investigador con peaje (autorizar y denegar) · bucle cerrado analista→investigador→analista
- [ ] **Arquitectura**: fork de decisiones en el mismo checkpoint · estabilidad · inyección de fallos · evaluación contra verdad determinista · contabilidad de costes

## VI.2 · Higiene al terminar

1. Esta campaña dejó **corridas de juguete** en la memoria (planes mínimos, experimentos dirigidos de prueba). Antes de tandas serias: limpia `memoria_unificada.json` o fija `DESDE` a la fecha de mañana.
2. Los hilos de los agentes viven en `MemorySaver` (RAM): mueren con el kernel — nada que limpiar.
3. Si activaste tracing sobre documentos sensibles, recuerda que los prompts están en LangSmith.

## VI.3 · El mapa que te llevas

La campaña ha recorrido las tres capas en su orden natural: el **gimnasio** produce evidencia bajo disciplina (examen congelado, remuestreo, estadística); el **observatorio** la agrega de forma determinista; los **agentes** la narran (analista) o proponen ampliarla (investigador) — siempre con el humano en los peajes y con cada número rastreable hasta una función de numpy. Cuando algo falle en producción, depura en ese orden: primero la evidencia, luego el cálculo, y solo al final la prosa.